# Plan

## Objective

Estimate, for each day of a 28-day window (Day 1 → Day 28), the number of eligible patients and the summary statistics of VFD-28 (ventilator-free days at 28 days) among mechanically ventilated patients — stratified by ICU type and by the year/ICU-type-specific pool of eligible providers — to support power calculations for a study of provider-level practice variation in mechanical ventilation management.

**Reference for VFD-28 definition & statistical guidance:** Yehya N, et al. *Reappraisal of Ventilator-Free Days in Critical Care Research.* Am J Respir Crit Care Med. 2019. https://pmc.ncbi.nlm.nih.gov/articles/PMC6812447/

## Cohort Eligibility (index criteria, evaluated only at the moment MV starts)

**Inclusion**
- Adults, `age_at_admission` ≥ 18
- Invasive mechanical ventilation (`device_category == IMV` in `respiratory_support`)
- First MV episode of the hospitalization only — if extubated and later reintubated, only the first episode counts. **Day 1 is always the first day of that first episode.**

**Exclusion (baked into the cohort — evaluated *at MV onset only*, never looking forward)**
- **ECMO at MV onset** (not ECMO at any point during the hospitalization). A patient cannulated onto ECMO on day 3 stays in — their Day-1 attending assignment was already legitimate when the day-1 trial happened.
- **Tracheostomy already in place at MV onset** (not tracheostomy placed later). Operationally: a trach documented within the first 24 hours is assumed to have been in place at onset.

**Flag columns only — NOT baked into the cohort exclusion.** Return as separate flags so the sample-size cost of each can be evaluated before committing to excluding them:
- **Cardiac arrest / anoxic brain injury, present on admission** — from `hospital_diagnosis`:
  - `flag_cardiac_arrest` — ICD-10 `I46.x`, only where `poa_present == 1`
  - `flag_anoxic_injury` — ICD-10 `G93.1`, only where `poa_present == 1`
  - Also return `diagnosis_primary` for comparison. POA-only is deliberate — an arrest coded without the POA flag could have happened on day 5 (after ventilation started) and should not exclude anyone.
- **Do-not-intubate status before MV initiation** — from `code_status`: take the status with the latest `start_dttm` at or before MV start.
  - `flag_dni` = 1 only for the three DNI-containing categories: **DNR/DNI, DNAR/DNI, DNI_only**
  - Plain **DNR, DNAR, and UDNR stay in** (not flagged) — in CLIF, DNR means "no CPR but do intubate," so these patients wanted the ventilator and belong in the study.

## VFD-28 Definition (Yehya et al. 2019 standard)

Computed once per patient's first vent episode, anchored to a fixed 28-day window starting Day 1 (vent initiation):

- **VFD-28 = 0** if the patient dies within 28 days of vent initiation (regardless of ventilation status at death)
- **VFD-28 = 0** if the patient is still on IMV at Day 28 (i.e., ventilated > 28 days)
- **VFD-28 = 28 − x** if successfully liberated from IMV on day x (extubated and off positive-pressure ventilation for > 48 hours without reintubation)
- **Reintubation:** if reintubated within 28 days, VFD-28 is counted from the day of the *final* successful extubation, not the first
- **Deaths after Day 28 are censored** (ignored) — only 28-day status matters
- **Discharge (placeholder, to revisit):** for now, censor at discharge the same way we censor at death/day-28 — i.e., no further ventilation status is tracked past discharge; VFD-28 is locked in based on in-hospital data as of the discharge date. This is a simplifying assumption pending further discussion, not a final decision.

**Day-anchoring (confirmed by Jared):** VFD-28 does **not** get recomputed for each day of the loop. Each patient has exactly one VFD-28 value, fixed relative to their own Day 1 — only the *set* of at-risk patients changes as the loop advances through Day 1 → Day 28.

**Required summary statistics per stratum:** mean, **SD** (this is the number the whole power calculation turns on — prioritize getting this right), median, IQR, proportion at 0, and proportion at 28.


## Provider Eligibility & ICU-type Stratification

A provider (attending, `ProvID`) is **eligible** for a given ICU type/year if they practiced (were active) in that `icu_type` during that year. Eligibility is nested by unit — a medical ICU patient could only ever have been covered by a medical ICU attending — so trials are stratified by (ICU type, year) and the counts must match that stratification.

**Two versions of "number of eligible providers," both needed:**

**(a) Per patient** — for each at-risk patient on each day, the count of eligible attendings they *could* have been assigned to that day (i.e., all providers active that year in that patient's `icu_type`). Report the **distribution** across patients — median and IQR is sufficient, no need for the raw list per patient.

**(b) Per ICU per year (roster table)**:

| year | hospital_id |icu_type    | eligible_providers (ProvID list) | N |
|------|--|-----------|------------------------------------|---|
| 2011 | ref | medical     | […]                                 | … |
| 2011 | 1 |surgical    | […]                                 | … |
| 2011 | 2 |neuro       | […]                                 | … |
| 2011 | 2 |cardiovascular | […]                              | … |
| …    | |…           | …                                    | … |

**Stratification scope:**
- **East Bank only:** break out by ICU type **and** year — medical, surgical, neuro, cardiovascular.
- **Community hospitals:** do **not** split by unit type — the same attending roster covers all their units, so report at the hospital level only (still by year).

Jared's rule of thumb governing all of this: the sample sizes actually used in the analysis are the ones we power on — so patient counts and provider counts must be reported at the same stratification granularity that will be used in the eventual trial-level analysis.


## Daily Landmark Workflow

For each Day *d* = 1, 2, …, 28, and separately for each stratum (East Bank: hospital × ICU type × year; community hospitals: hospital × year):

1. **Identify patients at risk on Day *d*** — a patient counts if, **at the start of Day *d***, they are:
   - Still alive
   - Still on the ventilator
   - Not yet discharged (censored at discharge, same treatment as death — see VFD-28 Definition)
   - Still in the unit (`icu_type`)
   - Covered by an eligible attending (`ProvID` active that year in that `icu_type`)
   - A member of the base cohort (adult, first MV episode, no ECMO at onset, no trach at onset)
2. **Count:**
   - N patients at risk (Day *d*, stratum)
   - N eligible providers per patient (Day *d*, stratum) — distribution (median, IQR) across the at-risk patients
   - N eligible providers per ICU/year — roster count from the lookup table (East Bank only; hospital-level for community sites)
3. **Compute VFD-28 summary statistics** for the Day-*d* at-risk patient set:
   - Mean, **SD**, median, IQR
   - Proportion at 0
   - Proportion at 28
4. Repeat steps 1–3 for every day 1–28, every stratum in scope.

This produces a series of 28 "landmark" snapshots per stratum — the at-risk patient *set* shrinks day to day (patients drop out on liberation, death, discharge, or transfer out of unit), but each patient's own VFD-28 value never changes once computed. A patient who exits the risk set on Day *d* (e.g., liberated) still contributes their fixed VFD-28 value to every day's summary stats through Day *d*, just not to Day *d*+1 onward.

**Why "still on the ventilator" is part of at-risk, not just death/discharge:** the daily count isn't tracking the outcome (that's already fixed) — it's tracking who is still exposed to an attending's active ventilator-management decision that day. Once liberated, there's no more vent decision to attribute to a provider, so the patient exits the *denominator* even though their VFD-28 value keeps flowing into every day's stats up through their last at-risk day.


## Deliverables

- **Cohort table:** eligible patients with `flag_cardiac_arrest`, `flag_anoxic_injury` (+ `diagnosis_primary`), and `flag_dni` columns attached (not pre-excluded) so each exclusion's sample-size cost can be evaluated
- **Roster lookup table:** eligible providers by (year, ICU type) — East Bank; by (year, hospital) — community
- **Daily table:** N patients at risk, by (day 1–28) × stratum
- **Daily table:** N eligible providers per patient (median, IQR distribution) and N eligible providers per ICU/year (roster count), by (day 1–28) × stratum
- **Daily table:** VFD-28 summary stats (mean, SD, median, IQR, proportion at 0, proportion at 28) for the Day-*d* at-risk cohort, by (day 1–28) × stratum
- All outputs aggregate-only (cell counts ≥ 10) per CLIF federated-analysis policy — no patient-level export


## Open Questions

**Deferred for now (placeholder decisions in place, revisit later):**
- **Discharge before Day 28** — *placeholder in place:* censor at discharge the same way we censor at death (no post-discharge follow-up assumed). Revisit whether this undercounts liberation for patients discharged shortly after extubation but before the 48h confirmation window closes.
- **Community hospital list & East Bank ICU-type list** — exact `icu_type` category values for medical/surgical/neuro/cardiovascular at East Bank, and which sites count as "community," to be confirmed later. Not blocking — build the pipeline generically by `icu_type`/`hospital` so the specific site list can be filled in later.


# Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import json
import time
import clifpy
import polars as pl
import duckdb

# Global Settings

In [ ]:
pd.set_option('display.max_columns', None)

os.makedirs('output_no_share', exist_ok=True)
os.makedirs('output_to_box', exist_ok=True)

con = duckdb.connect(database='output_no_share/cardiac_hospitalization_id.duckdb')

with open("config.json", "r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

clif_path = cfg["data_directory"]
print(f"CLIF filepath: {clif_path}")

file_type = cfg['filetype']
print(f'Filetype: {file_type}')

time_zone = cfg['timezone']
print(f'Timezone: {time_zone}')

site_name = cfg.get('site_name', 'site')
print(f'Site name: {site_name}')

con.execute(f"SET TimeZone = '{time_zone}'")

# File paths

In [ ]:
ext = file_type   # "parquet" or "csv"
hosp_diagnosis_path    = f"{clif_path}/clif_hospital_diagnosis.{ext}"
procedure_path         = f"{clif_path}/clif_patient_procedures.{ext}"
hospitalization_path   = f"{clif_path}/clif_hospitalization.{ext}"
dnr_path               = f"{clif_path}/clif_code_status.{ext}"
patient_path           = f"{clif_path}/clif_patient.{ext}"
vent_path              = f"{clif_path}/clif_respiratory_support.{ext}"
dialysis_path          = f"{clif_path}/clif_crrt_therapy.{ext}"
patient_diagnosis_path = f"{clif_path}/clif_patient_diagnosis.{ext}"
adt_path               = f"{clif_path}/clif_adt.{ext}"
vitals_path            = f"{clif_path}/clif_vitals.{ext}"
assessment_path        = f"{clif_path}/clif_patient_assessments.{ext}"
micro_culture_path     = f"{clif_path}/clif_microbiology_culture.{ext}"
labs_path              = f"{clif_path}/clif_labs.{ext}"
intermittent_med_path  = f"{clif_path}/clif_medication_admin_intermittent.{ext}"
continuous_med_path    = f"{clif_path}/clif_medication_admin_continuous.{ext}"
provider_path          = f"{clif_path}/clif_provider.{ext}"
ecmo_path              = f"{clif_path}/clif_ecmo_mcs.{ext}"

In [ ]:
REQUIRED = {
    hosp_diagnosis_path:  ["hospitalization_id", "diagnosis_code", "poa_present", "diagnosis_primary"],
    procedure_path:       ["hospitalization_id", "procedure_code", "procedure_billed_dttm"],
    hospitalization_path: ["hospitalization_id", "patient_id", "admission_dttm", "discharge_dttm",
                           "age_at_admission", "admission_type_name", "admission_type_category",
                           "discharge_category"],
    adt_path:             ["hospitalization_id", "hospital_id", "in_dttm", "out_dttm", "location_category"],
    patient_path:         ["patient_id", "sex_category", "race_category", "ethnicity_category",
                           "language_category", "death_dttm"],
    assessment_path:      ["hospitalization_id", "recorded_dttm", "assessment_category",
                           "numerical_value"],
    vent_path:            ["hospitalization_id", "recorded_dttm", "device_category", "tracheostomy"],
    dnr_path:              ["patient_id", "start_dttm", "code_status_category"],
    provider_path:         ["hospitalization_id", "provider_id", "start_dttm", "stop_dttm", "provider_role_category"],
    ecmo_path:              ["hospitalization_id", "recorded_dttm"],
}

errors = []
for path, required_cols in REQUIRED.items():
    name = os.path.basename(path)
    if not os.path.exists(path):
        errors.append(f"  ✗ MISSING FILE:  {name}")
        continue
    actual_cols = set(duckdb.sql(f"SELECT * FROM '{path}' LIMIT 0").columns)
    missing = [c for c in required_cols if c not in actual_cols]
    if missing:
        errors.append(f"  ✗ MISSING COLS:  {name} → {missing}")
    else:
        print(f"  ✓ {name}")

if errors:
    print("\nPRE-FLIGHT FAILED:")
    for e in errors:
        print(e)
    raise RuntimeError("Fix the above issues before running the notebook.")
else:
    print("\nAll required CLIF tables present and columns verified. Ready to run.")

# Pipeline Constants

All tunable rules from the Plan are centralized here — change once, applies everywhere below.

In [ ]:
ECMO_ONSET_BUFFER_HOURS  = 0     # ECMO recorded at/before MV start (+buffer) => "on ECMO at onset" (excluded).
                                  # Kept at 0 to strictly honor "evaluated only at MV onset, never looking
                                  # forward" — a >0 buffer would exclude patients cannulated onto ECMO shortly
                                  # AFTER MV started, which the eligibility rule explicitly says must stay in.
TRACH_ONSET_WINDOW_HOURS = 24    # tracheostomy flag within this window of MV start => "already in place at onset" (excluded)
LIBERATION_CONFIRM_HOURS = 48    # minimum sustained off-support duration to count as a successful extubation
VFD_WINDOW_DAYS          = 28

# DNI-containing code_status categories => flag_dni = 1 (plain DNR/DNAR/UDNR stay unflagged — see Cohort Eligibility cell)
DNI_CATEGORIES = ("DNR/DNI", "DNAR/DNI", "DNI_only")

# device_category values that represent being OFF positive-pressure support outright (liberated),
# regardless of tracheostomy status.
OFF_SUPPORT_DEVICES = ("Room Air", "Nasal Cannula", "Face Mask", "Trach Collar")

# Noninvasive support devices — per Yehya et al. 2019, these should NOT be held against liberation
# for a standard (non-tracheostomized) patient ("we recommend not counting noninvasive support"
# toward VFD; a patient extubated then bridged to NIV/HFNC is still credited as liberated from IMV).
# The stricter "off ALL positive-pressure support" standard is scoped by the article specifically to
# TRACHEOSTOMIZED patients ("tracheostomies should be treated as other invasive ventilation").
# So: at a given respiratory_support row, NIV_DEVICES count as OFF unless tracheostomy=1 at that row
# (i.e., the patient already has a trach in place, in which case NIV/HFNC use is treated as still-on
# support, same as IMV). IMV itself is always ON. Any other/unrecognized device_category is ON
# (conservative default).
NIV_DEVICES = ("NIPPV", "High Flow NC")

# ICU-type stratification is deferred (see Open Questions) — location_category='icu' is the
# only granularity available generically today. Once the East Bank medical/surgical/neuro/
# cardiovascular breakdown is finalized, update ICU_LOCATION_CATEGORIES and the icu_type
# column definition in the ADT join (Step 4) to key off `location_name` (or a mapping table)
# instead of the generic `location_category`.
ICU_LOCATION_CATEGORIES = ("icu",)

# Step 1 — Cohort Identification

Day 1 = the first ever IMV record for a hospitalization (the start of the first, and only-counted, MV episode). ECMO-at-onset and tracheostomy-at-onset are evaluated only in a window around that moment, never looking forward. Cardiac-arrest/anoxic-injury (POA) and DNI status are computed as flags, not exclusions.

In [ ]:
# --- 1a: Day 1 index — first-ever IMV record per hospitalization ---
mv_first = con.execute(f"""
    SELECT hospitalization_id, MIN(recorded_dttm) AS mv_start_dttm
    FROM '{vent_path}'
    WHERE device_category = 'IMV'
    GROUP BY hospitalization_id
""").df()
print(f"Hospitalizations with >=1 IMV record: {len(mv_first):,}")

# --- 1b: base cohort — join hospitalization + patient, apply age filter ---
con.register("mv_first", mv_first)
cohort_base = con.execute(f"""
    SELECT
        h.patient_id, h.hospitalization_id, m.mv_start_dttm,
        h.age_at_admission, h.admission_dttm, h.discharge_dttm, h.discharge_category,
        pt.death_dttm
    FROM mv_first m
    JOIN '{hospitalization_path}' h USING (hospitalization_id)
    JOIN '{patient_path}' pt USING (patient_id)
    WHERE h.age_at_admission >= 18
""").df()
print(f"After age >= 18 filter: {len(cohort_base):,}")

# --- 1c: ECMO-at-onset exclusion ---
con.register("cohort_base", cohort_base)
ecmo_onset_ids = set(con.execute(f"""
    SELECT b.hospitalization_id
    FROM cohort_base b
    JOIN '{ecmo_path}' e USING (hospitalization_id)
    GROUP BY b.hospitalization_id, b.mv_start_dttm
    HAVING MIN(e.recorded_dttm) <= b.mv_start_dttm + INTERVAL '{ECMO_ONSET_BUFFER_HOURS} hours'
""").df()["hospitalization_id"])
print(f"ECMO-at-onset excluded: {len(ecmo_onset_ids):,}")

# --- 1d: tracheostomy-at-onset exclusion ---
trach_onset_ids = set(con.execute(f"""
    SELECT DISTINCT b.hospitalization_id
    FROM cohort_base b
    JOIN '{vent_path}' r USING (hospitalization_id)
    WHERE r.tracheostomy = 1
      AND r.recorded_dttm BETWEEN b.mv_start_dttm AND b.mv_start_dttm + INTERVAL '{TRACH_ONSET_WINDOW_HOURS} hours'
""").df()["hospitalization_id"])
print(f"Tracheostomy-at-onset excluded: {len(trach_onset_ids):,}")

excluded_ids = ecmo_onset_ids | trach_onset_ids
cohort = cohort_base[~cohort_base["hospitalization_id"].isin(excluded_ids)].copy()
print(f"Base cohort after ECMO/trach-at-onset exclusions: {len(cohort):,} "
      f"(excluded {len(excluded_ids):,} of {len(cohort_base):,})")

In [ ]:
# --- 1e: flag columns (NOT exclusions) — POA cardiac arrest / anoxic injury, DNI status ---
# diagnosis_code formatting is inconsistent (some rows have the decimal, e.g. "I46.2",
# others don't, e.g. "G931") — REPLACE(...,'.','') normalizes both before matching.
con.register("cohort_ids", cohort[["hospitalization_id", "patient_id", "mv_start_dttm"]])

dx_flags = con.execute(f"""
    SELECT hospitalization_id,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') LIKE 'I46%' AND poa_present = 1
                     THEN 1 ELSE 0 END) AS flag_cardiac_arrest,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') LIKE 'I46%' AND poa_present = 1 AND diagnosis_primary = 1
                     THEN 1 ELSE 0 END) AS flag_cardiac_arrest_primary,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') LIKE 'G931%' AND poa_present = 1
                     THEN 1 ELSE 0 END) AS flag_anoxic_injury,
           MAX(CASE WHEN REPLACE(diagnosis_code, '.', '') LIKE 'G931%' AND poa_present = 1 AND diagnosis_primary = 1
                     THEN 1 ELSE 0 END) AS flag_anoxic_injury_primary
    FROM '{hosp_diagnosis_path}'
    WHERE hospitalization_id IN (SELECT hospitalization_id FROM cohort_ids)
    GROUP BY hospitalization_id
""").df()

# code_status is patient-level, but the "latest status at/before MV start" must be evaluated
# PER HOSPITALIZATION (a patient with multiple hospitalizations has a different mv_start_dttm,
# and therefore a different cutoff, for each) — so rank within (hospitalization_id), not patient_id,
# even though the join itself is on patient_id (code_status has no hospitalization_id of its own).
dni_flags = con.execute(f"""
    WITH ranked AS (
        SELECT c.hospitalization_id, cs.code_status_category,
               ROW_NUMBER() OVER (PARTITION BY c.hospitalization_id ORDER BY cs.start_dttm DESC) AS rn
        FROM '{dnr_path}' cs
        JOIN cohort_ids c ON c.patient_id = cs.patient_id AND cs.start_dttm <= c.mv_start_dttm
    )
    SELECT hospitalization_id,
           CASE WHEN code_status_category IN {DNI_CATEGORIES} THEN 1 ELSE 0 END AS flag_dni,
           code_status_category AS dni_source_category
    FROM ranked WHERE rn = 1
""").df()

cohort = cohort.merge(dx_flags, on="hospitalization_id", how="left")
cohort = cohort.merge(dni_flags, on="hospitalization_id", how="left")
for c in ["flag_cardiac_arrest", "flag_cardiac_arrest_primary", "flag_anoxic_injury", "flag_anoxic_injury_primary", "flag_dni"]:
    cohort[c] = cohort[c].fillna(0).astype(int)

print(f"flag_cardiac_arrest=1: {cohort['flag_cardiac_arrest'].sum():,}  "
      f"flag_anoxic_injury=1: {cohort['flag_anoxic_injury'].sum():,}  "
      f"flag_dni=1: {cohort['flag_dni'].sum():,}  "
      f"(hospitalizations with no code_status before MV start: {(cohort['dni_source_category'].isna()).sum():,})")
cohort.head()

# Step 2 — VFD-28 Computation

Follows Yehya et al. 2019: `x` = number of **full days elapsed** since MV start (Day 0 = initiation), `VFD28 = 28 - x`. This is why x uses `floor(elapsed_hours / 24)`, not a 1-indexed day label — using the 1-indexed "Day d" label here would make VFD28=28 mathematically unreachable (validated during testing: the off-by-one was caught by checking `proportion at 28` against synthetic data and finding it was always 0).

Reintubation is handled by walking the full device-transition sequence per patient and keeping only the **last** off-support period that has no subsequent reintubation before censoring — matching "counted from the day of final successful extubation."

Discharge censoring (placeholder decision) falls out naturally here: `censor_dttm = min(mv_start + 28 days, discharge_dttm)`, and the device-transition walk simply never sees data past that point.

In [ ]:
# --- 2a: window / censor timestamps ---
cohort["window_end_dttm"] = cohort["mv_start_dttm"] + pd.Timedelta(days=VFD_WINDOW_DAYS)
cohort["died_in_window"] = cohort["death_dttm"].notna() & (cohort["death_dttm"] <= cohort["window_end_dttm"])
cohort["censor_dttm"] = cohort[["window_end_dttm", "discharge_dttm"]].min(axis=1)

# --- 2b: pull in-window respiratory_support records for the cohort ---
con.register("cohort_censor", cohort[["hospitalization_id", "mv_start_dttm", "censor_dttm"]])
resp_window = con.execute(f"""
    SELECT r.hospitalization_id, r.recorded_dttm, r.device_category
    FROM '{vent_path}' r
    JOIN cohort_censor c ON c.hospitalization_id = r.hospitalization_id
    WHERE r.recorded_dttm >= c.mv_start_dttm AND r.recorded_dttm <= c.censor_dttm
    ORDER BY r.hospitalization_id, r.recorded_dttm
""").df()
resp_window["state"] = np.where(resp_window["device_category"].isin(OFF_SUPPORT_DEVICES), "OFF", "ON")
print(f"In-window respiratory_support records: {len(resp_window):,}")

# --- 2c: final-liberation walk per patient ---
censor_map = cohort.set_index("hospitalization_id")["censor_dttm"]

def find_final_liberation(group):
    """Walks the device-state sequence; returns the start of the LAST off-support period
    that has no subsequent reintubation before censoring (the 'final successful extubation'
    candidate), plus whether it met the 48h sustained-liberation threshold."""
    hosp_id = group.name
    censor = censor_map[hosp_id]
    off_start = None
    for _, row in group.sort_values("recorded_dttm").iterrows():
        if row["state"] == "OFF" and off_start is None:
            off_start = row["recorded_dttm"]
        elif row["state"] == "ON" and off_start is not None:
            off_start = None  # reintubated — this off-period doesn't count, keep looking
    if off_start is None:
        return pd.Series({"liberated_dttm": pd.NaT, "liberation_confirmed": False})
    duration_hours = (censor - off_start).total_seconds() / 3600.0
    return pd.Series({"liberated_dttm": off_start, "liberation_confirmed": duration_hours >= LIBERATION_CONFIRM_HOURS})

liberation = resp_window.groupby("hospitalization_id").apply(find_final_liberation).reset_index()
cohort = cohort.merge(liberation, on="hospitalization_id", how="left")
cohort["liberation_confirmed"] = cohort["liberation_confirmed"].fillna(False)

# --- 2d: VFD-28 ---
def elapsed_full_days(mv_start, t):
    return int(np.floor((t - mv_start).total_seconds() / 86400))

def compute_vfd28(row):
    if row["died_in_window"]:
        return 0
    if row["liberation_confirmed"]:
        x = min(elapsed_full_days(row["mv_start_dttm"], row["liberated_dttm"]), VFD_WINDOW_DAYS)
        return max(0, VFD_WINDOW_DAYS - x)
    return 0  # still ventilated at censor, or liberation not confirmed (<48h before censor)

cohort["vfd28"] = cohort.apply(compute_vfd28, axis=1)

print(cohort["vfd28"].describe())
print(f"\nProportion VFD28=0:  {(cohort['vfd28'] == 0).mean():.3f}")
print(f"Proportion VFD28=28: {(cohort['vfd28'] == 28).mean():.3f}")
print(f"Died in window:      {cohort['died_in_window'].sum():,}")
print(f"Liberation confirmed:{cohort['liberation_confirmed'].sum():,}")

# Step 3 — Provider Roster (year × ICU type)

A provider is eligible for a given (hospital, icu_type, year) if their Attending-role assignment overlapped an ICU-location ADT interval for that hospitalization, in that year. This is the lookup table sketched in the Teams thread.

In [ ]:
icu_cat_list = "(" + ", ".join(f"'{c}'" for c in ICU_LOCATION_CATEGORIES) + ")"

provider_roster = con.execute(f"""
    WITH prov_adt_overlap AS (
        SELECT
            prov.provider_id,
            adt.hospital_id,
            adt.location_category AS icu_type,
            GREATEST(prov.start_dttm, adt.in_dttm) AS overlap_start
        FROM '{provider_path}' prov
        JOIN '{adt_path}' adt USING (hospitalization_id)
        WHERE prov.provider_role_category = 'Attending'
          AND adt.location_category IN {icu_cat_list}
          AND prov.start_dttm < adt.out_dttm
          AND prov.stop_dttm > adt.in_dttm
    )
    SELECT hospital_id, icu_type, YEAR(overlap_start) AS year,
           COUNT(DISTINCT provider_id) AS n_eligible_providers
    FROM prov_adt_overlap
    GROUP BY hospital_id, icu_type, year
    ORDER BY hospital_id, icu_type, year
""").df()

print(f"Roster rows (hospital x icu_type x year): {len(provider_roster)}")
provider_roster

# Step 4 — Daily At-Risk Landmark Loop (Day 1–28)

For each patient × each Day *d*, at-risk requires: alive, not yet discharged, still on the ventilator (real-time device state — see note below), still in an ICU location, and covered by an Attending — evaluated at the **start** of Day *d* (`mv_start_dttm + (d-1) days`).

**Real-time state vs. VFD-28's "final liberation":** these are deliberately different computations. VFD-28 (Step 2) only cares about the *final* sustained liberation. The at-risk "still on the ventilator" check needs the *actual* moment-to-moment device state — including brief off-periods that later get reintubated (which don't count toward VFD-28, but which do mean the patient is genuinely off the vent, and therefore not at-risk, for those particular days). DuckDB's `ASOF JOIN` finds the most recent device-state record at-or-before each day's start.

In [ ]:
# --- 4a: real-time device-state timeline (distinct from Step 2's "final liberation" logic) ---
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE resp_state AS
    SELECT r.hospitalization_id, r.recorded_dttm,
           CASE WHEN r.device_category IN {tuple(OFF_SUPPORT_DEVICES)} THEN 'OFF' ELSE 'ON' END AS state
    FROM '{vent_path}' r
    JOIN cohort_censor c ON c.hospitalization_id = r.hospitalization_id
    WHERE r.recorded_dttm >= c.mv_start_dttm AND r.recorded_dttm <= c.censor_dttm
""")

# --- 4b: cohort x Day 1..28 scaffold ---
con.register("cohort_full", cohort)
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE days AS
    SELECT c.hospitalization_id, c.patient_id, c.mv_start_dttm, c.death_dttm,
           c.discharge_dttm, c.censor_dttm, c.vfd28,
           d.day,
           c.mv_start_dttm + (d.day - 1) * INTERVAL '1 day' AS day_start_dttm
    FROM cohort_full c
    CROSS JOIN generate_series(1, {VFD_WINDOW_DAYS}) AS d(day)
""")
print(f"Day-level rows (cohort x {VFD_WINDOW_DAYS} days): {con.execute('SELECT count(*) FROM days').fetchone()[0]:,}")

# --- 4c: ASOF join for real-time on/off state at each day's start ---
con.execute("""
    CREATE OR REPLACE TEMP TABLE days_state AS
    SELECT d.*, COALESCE(rs.state, 'ON') AS state_at_day_start
    FROM days d
    ASOF LEFT JOIN resp_state rs
      ON d.hospitalization_id = rs.hospitalization_id
     AND d.day_start_dttm >= rs.recorded_dttm
""")

# --- 4d: ADT (unit location) and provider (Attending coverage) intervals, filtered to cohort ---
adt_cohort = con.execute(f"""
    SELECT a.hospitalization_id, a.hospital_id, a.location_category, a.in_dttm, a.out_dttm
    FROM '{adt_path}' a
    JOIN cohort_full c ON c.hospitalization_id = a.hospitalization_id
""").df()
con.register("adt_cohort", adt_cohort)

prov_cohort = con.execute(f"""
    SELECT pr.hospitalization_id, pr.provider_id, pr.start_dttm, pr.stop_dttm
    FROM '{provider_path}' pr
    JOIN cohort_full c ON c.hospitalization_id = pr.hospitalization_id
    WHERE pr.provider_role_category = 'Attending'
""").df()
con.register("prov_cohort", prov_cohort)

# --- 4e: assemble at-risk flag ---
daily = con.execute("""
    SELECT
        ds.hospitalization_id, ds.patient_id, ds.day, ds.day_start_dttm, ds.vfd28,
        adt.hospital_id, adt.location_category AS icu_type,
        YEAR(ds.mv_start_dttm) AS index_year,
        (ds.death_dttm IS NULL OR ds.death_dttm > ds.day_start_dttm) AS is_alive,
        (ds.discharge_dttm IS NULL OR ds.discharge_dttm > ds.day_start_dttm) AS not_discharged,
        (ds.state_at_day_start = 'ON') AS still_on_vent,
        (adt.location_category IS NOT NULL) AS still_in_unit,
        (pr.provider_id IS NOT NULL) AS has_attending
    FROM days_state ds
    LEFT JOIN adt_cohort adt
      ON adt.hospitalization_id = ds.hospitalization_id
     AND ds.day_start_dttm >= adt.in_dttm AND ds.day_start_dttm < adt.out_dttm
     AND adt.location_category IN %s
    LEFT JOIN prov_cohort pr
      ON pr.hospitalization_id = ds.hospitalization_id
     AND ds.day_start_dttm >= pr.start_dttm AND ds.day_start_dttm < pr.stop_dttm
""" % icu_cat_list).df()

daily["at_risk"] = (
    daily["is_alive"] & daily["not_discharged"] & daily["still_on_vent"]
    & daily["still_in_unit"].fillna(False) & daily["has_attending"].fillna(False)
)

# --- 4f: attach per-patient eligible-provider count from the roster (indexed at MV-start year) ---
daily = daily.merge(
    provider_roster.rename(columns={"year": "index_year"}),
    on=["hospital_id", "icu_type", "index_year"], how="left",
)

print(f"at_risk rows: {daily['at_risk'].sum():,} / {len(daily):,}")
daily.groupby("day")["at_risk"].sum().head(10)

# Step 5 — Daily Summary Aggregation & Output

One row per (day, hospital, icu_type, year), computed only over that day's at-risk patients. Cells with n_at_risk < 10 are flagged for suppression before anything leaves the site (CLIF federated policy).

In [ ]:
at_risk = daily[daily["at_risk"]].copy()
con.register("at_risk", at_risk)

daily_summary = con.execute("""
    SELECT
        day, hospital_id, icu_type, index_year AS year,
        COUNT(*) AS n_at_risk,
        AVG(vfd28) AS vfd28_mean,
        STDDEV(vfd28) AS vfd28_sd,
        MEDIAN(vfd28) AS vfd28_median,
        QUANTILE_CONT(vfd28, 0.25) AS vfd28_q1,
        QUANTILE_CONT(vfd28, 0.75) AS vfd28_q3,
        AVG(CASE WHEN vfd28 = 0 THEN 1.0 ELSE 0 END) AS vfd28_prop_0,
        AVG(CASE WHEN vfd28 = 28 THEN 1.0 ELSE 0 END) AS vfd28_prop_28,
        MEDIAN(n_eligible_providers) AS elig_providers_per_pt_median,
        QUANTILE_CONT(n_eligible_providers, 0.25) AS elig_providers_per_pt_q1,
        QUANTILE_CONT(n_eligible_providers, 0.75) AS elig_providers_per_pt_q3,
        MAX(n_eligible_providers) AS elig_providers_roster_n
    FROM at_risk
    GROUP BY day, hospital_id, icu_type, index_year
    ORDER BY day, hospital_id, icu_type, index_year
""").df()

daily_summary["suppressed_lt10"] = daily_summary["n_at_risk"] < 10
print(f"Stratum-days with n_at_risk < 10: {daily_summary['suppressed_lt10'].sum()} / {len(daily_summary)}")

# --- write outputs ---
cohort.drop(columns=["window_end_dttm"]).to_parquet("output_no_share/cohort.parquet", index=False)
provider_roster.to_parquet("output_to_box/provider_roster.parquet", index=False)
daily_summary[~daily_summary["suppressed_lt10"]].to_csv("output_to_box/daily_summary.csv", index=False)
daily_summary.to_parquet("output_no_share/daily_summary_unsuppressed.parquet", index=False)

print("\nWrote: output_no_share/cohort.parquet, output_no_share/daily_summary_unsuppressed.parquet")
print("Wrote: output_to_box/provider_roster.parquet, output_to_box/daily_summary.csv (cells <10 suppressed)")
daily_summary.head(10)

# QC / Review Notes — synthetic-data test run

**Findings that need a team decision before this runs on real data:**

- **Roughly half of Day-1 MV starts happen before ICU arrival.** In the synthetic set, of 44,272 base-cohort patients, only 15,633 (~35%) are in an `icu`-category ADT location at the exact instant of their first IMV record — the rest are still in the ED (or ward/procedural). Since at-risk requires *simultaneously* being in an ICU location and having an eligible attending, this collapses the Day-1 at-risk count well below the full cohort size. **Open question for Nick/Jared:** should Day 1 eligibility be anchored to the moment of intubation (current implementation, patients intubated pre-ICU are excluded from Day 1 but can enter the at-risk set on whichever later day they arrive in the ICU), or to the moment of first ICU arrival after intubation? This materially changes the Day-1 denominator and hasn't been discussed yet.

**Known synthetic-data artifacts (do not generalize from these numbers — they're implementation smoke tests, not clinical findings):**

- `clif_provider.parquet` in this synthetic set assigns **exactly one uniquely-generated Attending per hospitalization** (e.g., `H352006-Attending`), with no provider ever repeating across patients, and the Attending span always equals the full admission-to-discharge window exactly. Real data should have a much smaller, recurring set of attendings per unit — the roster `n_eligible_providers` values seen here (in the tens of thousands) are a synthetic-generator artifact, not a real roster size. The join logic itself (Attending-span ∩ ICU-location-span) is correct and will produce realistic numbers once run against real provider data.
- No `I46.x` (cardiac arrest) or `G93.1` (anoxic injury) POA diagnoses exist in this synthetic set, so both flags read 0 — this validates the query runs cleanly, not that the logic is clinically exercised. Similarly, `code_status_category` only contains `Full`/`DNR/DNI`/`AND` here — `DNAR/DNI` and `DNI_only` are untested against real data.
- Single synthetic hospital (`HOSP0`) and a single generic `icu` location category — the East Bank medical/surgical/neuro/cardiovascular split and the community-hospital rollup (both deferred per the Plan) cannot be exercised until real ADT unit-naming is available.

**Validated correct during development (see conversation for worked examples):**
- Reintubation → VFD-28 anchored to the chronologically *final* successful extubation, ignoring brief earlier off-periods that were followed by reintubation (confirmed against `hospitalization_id=352255`: extubated → reintubated 1h later → extubated again and never reintubated → VFD-28 correctly anchors to the second extubation).
- Death within the 28-day window overrides any device-state complexity to VFD-28=0, even with a tangled reintubation history (confirmed against `hospitalization_id=6`).
- The 48-hour liberation-confirmation threshold correctly zeroes out extubations that are followed too soon by discharge/censoring to confirm sustained liberation (confirmed against `hospitalization_id=352006`: extubated, but discharged only 24h later — VFD-28=0, not credited).
- `n_eligible_providers` computed per-patient (Step 4f) exactly matches the roster lookup (Step 3) within each stratum — expected, since both are keyed off the same (hospital, icu_type, year), and serves as an internal consistency check.